# 02 — Rentabilidad y comportamiento de compra (A2)
Pregunta central: ¿dónde gana y dónde pierde margen RetailVenture?
Bloques:
1. Margen por categoría, canal y segmento
2. Efecto real de los descuentos
3. CLV por segmento y cohortes
4. Estacionalidad
Fuente: data/clean/ (salida del notebook 01).

## Bloque 1 — Margen por categoría, canal y segmento
Margen bruto = monto_neto − cantidad × costo del producto.

In [1]:
import pandas as pd
from pathlib import Path

CLEAN = Path("../data/clean")
cli = pd.read_csv(CLEAN / "clientes.csv", parse_dates=["fecha_registro", "primera_compra", "ultima_compra", "fecha_alta_efectiva"])
tx = pd.read_csv(CLEAN / "transacciones.csv", parse_dates=["fecha"])
pr = pd.read_csv(CLEAN / "productos.csv")

t = (tx.merge(pr[["producto_id", "categoria", "costo"]], on="producto_id")
       .merge(cli[["cliente_id", "segmento"]], on="cliente_id"))
t["margen"] = t.monto_neto - t.cantidad * t.costo

def resumen(dim):
    r = t.groupby(dim).agg(ventas=("monto_neto", "sum"), margen=("margen", "sum"),
                           transacciones=("transaccion_id", "count"), descuento_prom=("descuento", "mean"))
    r["margen_pct"] = r.margen / r.ventas * 100
    r["pct_ventas"] = r.ventas / r.ventas.sum() * 100
    r["pct_margen"] = r.margen / r.margen.sum() * 100
    r["descuento_prom"] *= 100
    return r.sort_values("ventas", ascending=False).round(1)

print(f"Ventas: ${t.monto_neto.sum():,.0f} | Margen: ${t.margen.sum():,.0f} "
      f"({t.margen.sum() / t.monto_neto.sum():.1%}) | Transacciones a pérdida: {(t.margen < 0).sum()}")

Ventas: $17,392,987 | Margen: $4,310,329 (24.8%) | Transacciones a pérdida: 711


In [2]:
resumen("categoria")

,ventas,margen,transacciones,descuento_prom,margen_pct,pct_ventas,pct_margen
categoria,,,,,,,
Electrónica,7662506.1,1049301.2,8419,5.3,13.7,44.1,24.3
Hogar_y_Deco,4465865.0,1353038.1,8355,5.1,30.3,25.7,31.4
Deportes,2592709.1,907213.7,8311,5.4,35.0,14.9,21.0
Ropa_y_Calzado,1513396.7,597005.3,8311,5.1,39.4,8.7,13.9
Belleza_y_Salud,748631.6,309502.6,8287,5.2,41.3,4.3,7.2
Alimentos_y_Bebidas,409879.0,94268.1,8317,5.1,23.0,2.4,2.2


In [3]:
resumen("canal")

,ventas,margen,transacciones,descuento_prom,margen_pct,pct_ventas,pct_margen
canal,,,,,,,
tienda_fisica,7862504.3,2046091.6,22343,3.7,26.0,45.2,47.5
online,6547309.2,1491185.5,19220,7.5,22.8,37.6,34.6
app_movil,2983174.0,773051.9,8437,3.7,25.9,17.2,17.9


In [4]:
resumen("segmento")

,ventas,margen,transacciones,descuento_prom,margen_pct,pct_ventas,pct_margen
segmento,,,,,,,
Frecuente,8766759.1,2131971.0,25458,5.6,24.3,50.4,49.5
Premium,5280660.8,1345562.6,14907,4.2,25.5,30.4,31.2
Esporádico,2701501.9,674499.2,7792,5.7,25.0,15.5,15.6
Nuevo,326779.1,80590.2,962,5.5,24.7,1.9,1.9
Inactivo,317286.6,77706.0,881,5.8,24.5,1.8,1.8


**Hallazgos:**
- **Electrónica vende mucho y deja poco**: 44% de las ventas, pero solo 24% del margen.
  Hogar y Deco hace lo contrario: 26% de las ventas y 31% del margen.
- **El canal online descuenta el doble** que la tienda física y por eso tiene peor margen
  (23% contra 26%).
- **Por segmento, los márgenes son parejos (24–26%)**: la diferencia está en el volumen.
  Frecuente y Premium generan el 81% del margen.
- **711 transacciones se venden a pérdida.** Se analizan en el bloque 2.

## H1.1 — Mezcla de productos
Hipótesis: el ticket promedio bajó en 2024 porque los clientes compraron productos más baratos.
Si fuera así, la proporción de ventas de productos caros debería caer entre 2023 y 2024.

In [5]:
# Terciles de productos por precio de lista (tx y pr ya se cargaron en el bloque 1)
productos_t = pr.copy()
productos_t["tercil"], cortes = pd.qcut(productos_t.precio_lista, 3, labels=["Barato", "Medio", "Caro"], retbins=True)
print(f"Cortes: ${cortes[1]:,.2f} y ${cortes[2]:,.2f}")

mezcla = tx.merge(productos_t[["producto_id", "tercil"]], on="producto_id")
mezcla["anio"] = mezcla.fecha.dt.year

Cortes: $57.87 y $200.35


In [6]:
pct_tx = pd.crosstab(mezcla.tercil, mezcla.anio, normalize="columns") * 100
pct_monto = mezcla.pivot_table(index="tercil", columns="anio", values="monto_neto", aggfunc="sum", observed=True)
pct_monto = pct_monto / pct_monto.sum() * 100
ticket = mezcla.pivot_table(index="tercil", columns="anio", values="monto_neto", aggfunc="mean", observed=True)
pd.concat({"% transacciones": pct_tx, "% monto_neto": pct_monto, "ticket promedio": ticket}, axis=1).round(2)

% transacciones        % monto_neto        ticket promedio        
anio              2023   2024         2023   2024            2023    2024
tercil                                                                   
Barato           33.37  33.06         5.02   5.07           52.86   52.82
Medio            33.04  33.70        19.07  19.86          202.75  203.03
Caro             33.59  33.24        75.91  75.07          793.79  777.98

In [7]:
# Ticket 2024 si se hubiera mantenido la mezcla de 2023
ticket_anio = mezcla.groupby("anio").monto_neto.mean()
ticket_2024_mezcla_2023 = (pct_tx[2023] / 100 * ticket[2024]).sum()
print(f"Ticket 2023: ${ticket_anio[2023]:,.2f} | Ticket 2024: ${ticket_anio[2024]:,.2f} | "
      f"Ticket 2024 con mezcla 2023: ${ticket_2024_mezcla_2023:,.2f}")

Ticket 2023: $351.29 | Ticket 2024: $344.49 | Ticket 2024 con mezcla 2023: $346.06


In [8]:
# Tercil Caro: qué cambió dentro del tercil
caro = mezcla[mezcla.tercil == "Caro"].groupby("anio").agg(
    ticket=("monto_neto", "mean"), descuento_pct=("descuento", "mean"),
    cantidad=("cantidad", "mean"), precio_unitario=("precio_unitario", "mean"))
caro["descuento_pct"] *= 100
caro.round(2)

,ticket,descuento_pct,cantidad,precio_unitario
anio,,,,
2023,793.79,5.23,1.81,463.89
2024,777.98,5.35,1.78,462.17


**H1.1 descartada: la mezcla no cambió.**
- Con la mezcla de 2023, el ticket de 2024 habría sido $346,06 en vez de $344,49: la mezcla
  explica solo $1,57 de la caída de $6,80.
- La caída está dentro del tercil Caro: su ticket baja de $793,79 a $777,98 y su descuento
  medio sube de 5,23% a 5,35%. Eso apunta a los descuentos (H1.2), no a qué se compra.

## H1.2 — Descuentos
Hipótesis: los descuentos se comen el margen sin traer clientes más fieles.

In [9]:
# monto_neto debe cumplir precio_unitario × cantidad × (1 − descuento)
d = (tx.merge(pr[["producto_id", "categoria", "costo"]], on="producto_id")
       .merge(cli[["cliente_id", "segmento"]], on="cliente_id"))
cumple = (d.precio_unitario * d.cantidad * (1 - d.descuento) - d.monto_neto).abs() <= 0.02
print(f"Filas que cumplen la fórmula: {cumple.mean():.1%} de {len(d):,}")

d["margen"] = d.monto_neto - d.costo * d.cantidad
d["tramo"] = pd.cut((d.descuento * 100).round(6), [-0.01, 0, 5, 10, 15, 20, 100],
                    labels=["0", "1–5", "6–10", "11–15", "16–20", ">20"])

Filas que cumplen la fórmula: 100.0% de 50,000


In [10]:
tramos = d.groupby("tramo", observed=True).agg(
    ventas=("transaccion_id", "count"), unidades_prom=("cantidad", "mean"),
    margen_prom=("margen", "mean"), margen_total=("margen", "sum"),
    ventas_a_perdida=("margen", lambda m: (m < 0).sum()))
print(f"Ventas a pérdida: {tramos.ventas_a_perdida.sum()}")
tramos.round(2)

Ventas a pérdida: 711


,ventas,unidades_prom,margen_prom,margen_total,ventas_a_perdida
tramo,,,,,
0,14660,1.78,105.28,1543355.07,0
1–5,16922,1.78,96.35,1630490.71,0
6–10,8903,1.78,77.81,692767.37,5
11–15,4269,1.78,61.37,262004.83,102
16–20,3368,1.80,42.75,143988.13,269
>20,1878,1.79,20.09,37722.94,335


In [11]:
# 0% contra >20% de descuento, por categoría y por canal
extremos = d[d.tramo.isin(["0", ">20"])]

def comparar(dim):
    return extremos.pivot_table(index=dim, columns="tramo", observed=True,
                                values=["cantidad", "margen", "transaccion_id"],
                                aggfunc={"cantidad": "mean", "margen": "mean", "transaccion_id": "count"}
                                ).rename(columns={"cantidad": "unidades_prom", "margen": "margen_prom",
                                                  "transaccion_id": "ventas"}).round(2)

comparar("categoria")

unidades_prom       margen_prom        ventas     
tramo                           0   >20           0    >20      0  >20
categoria                                                             
Alimentos_y_Bebidas          1.79  1.83       13.79   2.14   2422  300
Belleza_y_Salud              1.81  1.67       42.54  18.64   2411  293
Deportes                     1.78  1.80      124.86  52.12   2406  343
Electrónica                  1.76  1.81      171.79 -45.32   2484  343
Hogar_y_Deco                 1.80  1.82      194.23  58.84   2484  299
Ropa_y_Calzado               1.75  1.80       80.65  38.98   2453  300

In [12]:
comparar("canal")

unidades_prom       margen_prom        ventas      
tramo                     0   >20           0    >20      0   >20
canal                                                            
app_movil              1.76  1.79      102.45  22.71   3639   131
online                 1.84  1.81      108.16  19.52   1222  1418
tienda_fisica          1.78  1.70      105.96  21.48   9799   329

In [13]:
# Quién recibe descuentos: % de ventas con descuento y descuento promedio
def quien(dim):
    r = d.groupby(dim).agg(pct_con_descuento=("descuento", lambda s: (s > 0).mean() * 100),
                           descuento_prom=("descuento", lambda s: s.mean() * 100))
    return r.round(1)

pd.concat({dim: quien(dim) for dim in ["canal", "categoria", "cantidad", "metodo_pago", "segmento"]})

pct_con_descuento  descuento_prom
canal       app_movil                         56.9             3.7
            online                            93.6             7.5
            tienda_fisica                     56.1             3.7
categoria   Alimentos_y_Bebidas               70.9             5.1
            Belleza_y_Salud                   70.9             5.2
            Deportes                          71.1             5.4
            Electrónica                       70.5             5.3
            Hogar_y_Deco                      70.3             5.1
            Ropa_y_Calzado                    70.5             5.1
cantidad    1                                 70.7             5.2
            2                                 70.5             5.1
            3                                 71.0             5.3
            4                                 70.0             5.2
            5                                 71.8             5.4
metodo_pago billetera_digital                 71.4             5.4
            efectivo                          71.2             5.2
            tarjeta_credito                   70.7             5.1
            tarjeta_debito                    70.3             5.2
            transferencia                     70.2             5.2
segmento    Esporádico                        76.0             5.7
            Frecuente                         74.8             5.6
            Inactivo                          74.2             5.8
            Nuevo                             76.2             5.5
            Premium                           60.3             4.2

In [14]:
# Primera compra de cada cliente contra las siguientes
orden = d.sort_values(["cliente_id", "fecha", "transaccion_id"])
orden["es_primera"] = ~orden.duplicated("cliente_id")
orden.groupby("es_primera").descuento.agg(
    pct_con_descuento=lambda s: (s > 0).mean() * 100,
    descuento_prom=lambda s: s.mean() * 100,
    pct_mas_de_15=lambda s: (s > 0.15).mean() * 100).rename(index={True: "primera", False: "siguientes"}).round(1)

,pct_con_descuento,descuento_prom,pct_mas_de_15
es_primera,,,
siguientes,70.1,5.1,10.3
primera,73.9,5.6,11.3


In [15]:
# Recompra según el descuento de la primera compra
primera = orden[orden.es_primera].set_index("cliente_id")
compras = orden.groupby("cliente_id").size()
recompra = pd.DataFrame({
    "tramo_primera": pd.cut((primera.descuento * 100).round(6), [-0.01, 0, 5, 15, 100],
                            labels=["0", "1–5", "6–15", ">15"]),
    "compras": compras})
recompra.groupby("tramo_primera", observed=True).agg(
    clientes=("compras", "size"), pct_vuelve=("compras", lambda n: (n > 1).mean() * 100),
    compras_prom=("compras", "mean")).round(1)

,clientes,pct_vuelve,compras_prom
tramo_primera,,,
0,1985,79.2,7.4
1–5,2590,78.3,6.5
6–15,2174,77.1,6.2
>15,859,75.4,5.7


In [16]:
# Total descontado = lo que se dejó de cobrar: monto bruto − monto_neto
d["bruto"] = d.precio_unitario * d.cantidad
descontado = (d.bruto - d.monto_neto).sum()
por_formula = (d.bruto * d.descuento).sum()
mas_15 = d.descuento > 0.15
print(f"Total descontado: ${descontado:,.0f} ({descontado / d.margen.sum():.1%} del margen total)")
print(f"Con precio × cantidad × descuento: ${por_formula:,.2f} (diferencia ${descontado - por_formula:,.2f})")
print(f"Descuentos >15%: ${(d.bruto - d.monto_neto)[mas_15].sum():,.0f} en {mas_15.sum():,} ventas")

Total descontado: $967,701 (22.5% del margen total)
Con precio × cantidad × descuento: $967,699.89 (diferencia $1.31)
Descuentos >15%: $386,906 en 5,246 ventas


**H1.2 confirmada: los descuentos se comen el margen y no fidelizan.**
- A más descuento, menos margen por venta: de $105,28 sin descuento a $20,09 con más de 20%.
  Las 711 ventas a pérdida están todas en tramos de 6% o más; 335 en el tramo >20%.
- Electrónica con más de 20% de descuento pierde $45,32 por venta.
- Se dejaron de cobrar $967.701; las ventas con más de 15% de descuento suman $386.906 en 5.246 ventas.
- El descuento no compra lealtad: quien entra con descuento vuelve menos (75,4% con >15% contra
  79,2% sin descuento) y compra menos veces (5,7 contra 7,4).
- La primera compra lleva descuento más seguido que las siguientes (73,9% contra 70,1%).

*Nota:* con precio × cantidad × descuento da $967.699,89; la diferencia de $1,31 viene del
redondeo a centavos de monto_neto en cada fila.